Refer to https://github.com/GreenleafLab/HDMA/blob/main/code/05-misc/04-ChromBPNet_use_cases.ipynb

In [ ]:
%load_ext autoreload
%autoreload 2

# deal with matplotlib plots
%matplotlib inline

# display all outputs in a cell
get_ipython().ast_node_interactivity = 'all'

from IPython.display import display, HTML

In [ ]:
# misc
import sys
import os
import numpy as np
import random
from tqdm import tqdm
import pandas as pd

# io
import h5py
from pyfaidx import Fasta

# ML
import bpnetlite
from bpnetlite import ChromBPNet
from bpnetlite import BPNet
from bpnetlite.bpnet import CountWrapper
from bpnetlite.attribute import deep_lift_shap

import torch

import pickle

import pyranges as pr

In [ ]:
# plotting
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn; seaborn.set_style('white')
import matplotlib.ticker as ticker
import plotnine as pn
import logomaker
import tangermeme
from tangermeme.plot import plot_logo

from tangermeme.utils import random_one_hot, _validate_input
from tangermeme.utils import characters
from tangermeme.ersatz import substitute
from tangermeme.utils import one_hot_encode

print(np.__version__)
print(tangermeme.__version__)
print(bpnetlite.__version__)
print(torch.__version__)

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES']='0'
#os.environ['CUDA_VISIBLE_DEVICES']='MIG-40f43250-998e-586a-ac37-d6520e92590f'

torch.manual_seed(100)

# editable text in PDFs
# https://stackoverflow.com/a/54111532
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

sys.path.append("~/HDMA/code/03-chrombpnet")
import tangermeme_utils as tanutils
from tangermeme_utils.utils import plot_motif
from tangermeme_utils.eval import summarize_across_folds, mean_across_sequences
from tangermeme_utils.wrappers import ChromBPNetWrapper, reshape_folds

In [ ]:
torch.cuda.device_count()

In [ ]:
dev = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
dev

torch.cuda.is_available()

t1 = torch.randn(1,2).to(dev)
t1
t1.device

In [ ]:
INPUTLEN = 2114
OUTPUTLEN = 1000
SHIFT_REL_INPUT = np.int64((2114 - 1000) / 2)
SHIFT_REL_INPUT

NARROWPEAK_SCHEMA = ["chr", "start", "end", "1", "2", "3", "4", "5", "6", "summit"]

LOGO_ALPHABET = 'ACGT'

# finemo colorscheme
LOGO_COLORS = {"A": '#109648', "C": '#255C99', "G": '#F7B32B', "T": '#D62839'}

# logomaker colorscheme
LOGO_COLORS2= {
        'G': [1, .65, 0],
        'T': [1, 0, 0],
        'C': [0, 0, 1],
        'A': [0, .5, 0]
    }

CHAR_IGNORE = ['QWERYUIOPSDFHJKLZXVBNM']

In [ ]:
def predict_sequence(genome,chrom,start,end,subtype,models,models_counts,min_pos=557,max_pos=1557,show_fig=False):

    mid=(end+start)//2

    input_start=mid - INPUTLEN//2
    input_end=mid + INPUTLEN//2
    pre_seq = str(genome[chrom][mid - INPUTLEN//2 : mid + INPUTLEN//2])
    assert len(pre_seq) == 2114
    output_start=mid - OUTPUTLEN//2
    output_end=mid + OUTPUTLEN//2
    

    #使用 .upper() 转换为大写
    seq_ohe = one_hot_encode(pre_seq.upper()).type(torch.float32).unsqueeze(0)

    # predict
    print("@ predicting...")
    seq_pred_list = []

    for model in models:
        y_seq = tangermeme.predict.predict(model, seq_ohe)
        seq_pred_list.append(y_seq)

    seq_pred = reshape_folds(seq_pred_list)

    seq_pred_profile_mean, _ = summarize_across_folds(seq_pred[0])

    seq_pred_profile_mean.shape

    # compute attribution
    print("@ computing attribution...")
    try:

        seq_attr_list = []
        
        for model in models_counts:
            y_seq = deep_lift_shap(model, seq_ohe, random_state=0)
            seq_attr_list.append(y_seq)

        seq_attr = torch.stack(seq_attr_list, dim=0)
        seq_attr_mean, _ = summarize_across_folds(seq_attr)

        
    except Exception as e:
        print(f"Error during attribution computation: {e}")

    results={"seq_pred_profile_mean":seq_pred_profile_mean,
        "seq_attr_mean":seq_attr_mean}

    pred_seq = results['seq_pred_profile_mean'].squeeze()

    attr_seq = results['seq_attr_mean'].sum(axis = 1).squeeze().T[(INPUTLEN // 2 - 500):(INPUTLEN // 2 + 500)]


    # ploting-1
    fig, ax = plt.subplots(3, 1, figsize = (25, 6))
    alpha = 0.3
    # prediction in 1,000 bp
    ax[0].plot(pred_seq, linewidth = 0.5, label = "seq(5-fold mean)", color = "blue")
    ax[0].set_ylabel("Pred. \naccessibility")
    ax[0].axvspan(250, 751, alpha = alpha, color='yellow')
    ax[0].legend(fontsize = 8)
    # attribution in 1,000 bp
    ax[1].plot(attr_seq, linewidth = 0.5, label = "seq(5-fold mean)", color = "blue")
    ax[1].set_ylabel("Contrib. \nscores")
    ax[1].axvspan(250, 751, alpha = alpha, color = 'yellow')
    ax[1].legend(fontsize = 8)
    #
    ylim = None
    tangermeme.plot.plot_logo(np.float64(results['seq_attr_mean'][0]), start = min_pos, end = max_pos, ax=ax[2], ylim = ylim)
    ax[2].set_ylabel("Contrib. \n")
        # put a title at the very top
    plt.suptitle(subtype + ":" + chrom + ":"+ str(start+1) + "-" + str(end), fontsize = 12)
    plt.xlabel("Genomic position (bp)")
    plt.tight_layout()
    
    if show_fig:
        plt.show()
    else:
        plt.savefig(os.path.join(outdir, subtype + ":" + chrom + ":"+ str(start+1) + "-" + str(end) + ".pdf"))

    plt.close(fig)
    
    '''
    ## ploting-2
    import logomaker
    attr_data = results['seq_attr_mean'][0].detach().cpu().numpy()
    attr_data_T = attr_data.T
    df = pd.DataFrame(attr_data_T, columns=['A', 'C', 'G', 'T'])
    crp_logo = logomaker.Logo(df, 
                          figsize=(120, 6), # 2114bp 建议先设宽一点
                          shade_below=0,
                          fade_below=0)
    ### ploting-3
    min_pos=min_pos
    max_pos=max_pos
    df_trimmed = df.loc[min_pos:max_pos,:]
    crp_logo = logomaker.Logo(df_trimmed, 
                            figsize=(25, 2), # 2114bp 建议先设宽一点
                            shade_below=0,
                            fade_below=0)
    '''

#human model predict

#B cell model

In [ ]:
proj_in="~/chrombpnet/chrombpnet_use_snakefile/Hs_C10white_blood_cells/output/models"

In [ ]:
#load model
models = [
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C10.1/models/chrombpnet_nobias.h5")))
]

models_counts = [
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C10.1/models/chrombpnet_nobias.h5")))
]

In [ ]:
#load model
#models = [
 #   ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C10.1/models/chrombpnet_nobias.h5"))),
  #  ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_1/C10.1/models/chrombpnet_nobias.h5"))),
    #ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_2/C10.1/models/chrombpnet_nobias.h5"))),
   # ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_3/C10.1/models/chrombpnet_nobias.h5"))),
    #ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_4/C10.1/models/chrombpnet_nobias.h5")))
#]

#models_counts = [
 #   CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C10.1/models/chrombpnet_nobias.h5"))),
  #  CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_1/C10.1/models/chrombpnet_nobias.h5"))),
   # CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_2/C10.1/models/chrombpnet_nobias.h5"))),
    #CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_3/C10.1/models/chrombpnet_nobias.h5"))),
    #CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_4/C10.1/models/chrombpnet_nobias.h5")))
#]

#human sequences

In [ ]:
outdir="~/chrombpnet/Hs_TS_immune_chrombpnet/B_cell_Ery_ortho_predict/Hs_B_pre_Hs_B"

In [ ]:
Hg38_genome_fa = os.path.join("~/chrombpnet/chrombpnet_tutorial/data/downloads/hg38.fa")

In [ ]:
hg38 = Fasta(Hg38_genome_fa)

In [ ]:
# orth_index94458 ------------
chrom="chr6"
start=41974289
end=41974790
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(hg38,chrom,start,end,subtype="C10.1",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr17:2,239,095-2,241,207

In [ ]:
# orth_index94453 ------------------------------------------
chrom="chr6"
start=42010300
end=42010801
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(hg38,chrom,start,end,subtype="C10.1",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr2:46564850-46565200

In [ ]:
# orth_index94451 ------------------------------------------
chrom="chr6"
start=42022288
end=42022789
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(hg38,chrom,start,end,subtype="C10.1",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr2:46564850-46565200

In [ ]:
# orth_index94450 ------------------------------------------
chrom="chr6"
start=42025414
end=42025915
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(hg38,chrom,start,end,subtype="C10.1",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr2:46564850-46565200

In [ ]:
# orth_index65090 ------------------------------------------
chrom="chr16"
start=28930678
end=28931179
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(hg38,chrom,start,end,subtype="C10.1",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr2:46564850-46565200

#tree shrew sequence

In [ ]:
outdir="~/chrombpnet/Hs_TS_immune_chrombpnet/B_cell_Ery_ortho_predict/Hs_B_pre_TS_B"

In [ ]:
TS_genome_fa = os.path.join("~/software/ref_tree_shrew/TreeShrew_scATAC_ref/fasta/genome.fa")

# output paths
#figout = os.path.join("~/chrombpnet/chromnpnet_cardiomyocytes/peak_from_ArchR_1000bp/chrombpnet_model_predicting/variant")

In [ ]:
TS_genome = Fasta(TS_genome_fa)

In [ ]:
# orth_index94458 ------------
chrom="chr3"
start=117496821
end=117497322
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(TS_genome,chrom,start,end,subtype="C10.1",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr17:2,239,095-2,241,207

In [ ]:
# orth_index94453 ------------------------------------------
chrom="chr3"
start=117454868
end=117455369
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(TS_genome,chrom,start,end,subtype="C10.1",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr2:46564850-46565200

In [ ]:
# orth_index94451 ------------------------------------------
chrom="chr3"
start=117441918
end=117442419
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(TS_genome,chrom,start,end,subtype="C10.1",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr2:46564850-46565200

In [ ]:
# orth_index94450 ------------------------------------------
chrom="chr3"
start=117438736
end=117439237
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(TS_genome,chrom,start,end,subtype="C10.1",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr2:46564850-46565200

In [ ]:
# orth_index65090 ------------------------------------------
chrom="chr2"
start=59849315
end=59849816
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(TS_genome,chrom,start,end,subtype="C10.1",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr2:46564850-46565200

红细胞

In [ ]:
proj_in="~/chrombpnet/chrombpnet_use_snakefile/Hs_C3Erythroid/output/models"

In [ ]:
#load model
models = [
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C3.1/models/chrombpnet_nobias.h5")))
]

models_counts = [
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C3.1/models/chrombpnet_nobias.h5")))
]

human sequences

In [ ]:
outdir="~/chrombpnet/Hs_TS_immune_chrombpnet/B_cell_Ery_ortho_predict/Hs_Ery_pre_Hs_Ery"

In [ ]:
# orth_index94462 ------------
chrom="chr6"
start=41957118
end=41957619
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(hg38,chrom,start,end,subtype="C3.1",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr17:2,239,095-2,241,207

In [ ]:
# orth_index94452 ------------
chrom="chr6"
start=42016971
end=42017472
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(hg38,chrom,start,end,subtype="C3.1",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr17:2,239,095-2,241,207

In [ ]:
# orth_index65100 ------------
chrom="chr16"
start=29009192
end=29009693
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(hg38,chrom,start,end,subtype="C3.1",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr17:2,239,095-2,241,207

### Tree shrew sequences


In [ ]:
outdir="~/chrombpnet/Hs_TS_immune_chrombpnet/B_cell_Ery_ortho_predict/Hs_Ery_pre_TS_Ery"

In [ ]:
# orth_index94462 ------------
chrom="chr3"
start=117448428
end=117448929
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(TS_genome,chrom,start,end,subtype="C3.1",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr17:2,239,095-2,241,207

In [ ]:
# orth_index94452 ------------
chrom="chr3"
start=117518919
end=117519420
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(TS_genome,chrom,start,end,subtype="C3.1",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr17:2,239,095-2,241,207

In [ ]:
# orth_index65100 ------------
chrom="chr2"
start=59909670
end=59910171
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(TS_genome,chrom,start,end,subtype="C3.1",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr17:2,239,095-2,241,207

#tree shrew model

#B cell

In [ ]:
proj_in="~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C24white_blood_cells/output/models"

In [ ]:
#load model
models = [
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C24.21/models/chrombpnet_nobias.h5")))
]

models_counts = [
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C24.21/models/chrombpnet_nobias.h5")))
]

#human sequences

In [ ]:
outdir="~/chrombpnet/Hs_TS_immune_chrombpnet/B_cell_Ery_ortho_predict/TS_B_pre_Hs_B"

In [ ]:
Hg38_genome_fa = os.path.join("~/chrombpnet/chrombpnet_tutorial/data/downloads/hg38.fa")

# output paths
#figout = os.path.join("~/chrombpnet/chromnpnet_cardiomyocytes/peak_from_ArchR_1000bp/chrombpnet_model_predicting/variant")

In [ ]:
hg38 = Fasta(Hg38_genome_fa)

In [ ]:
# orth_index94458 ------------
chrom="chr6"
start=41974289
end=41974790
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(hg38,chrom,start,end,subtype="C24.21",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr17:2,239,095-2,241,207

In [ ]:
# orth_index94453 ------------------------------------------
chrom="chr6"
start=42010300
end=42010801
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(hg38,chrom,start,end,subtype="C24.21",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr2:46564850-46565200

In [ ]:
# orth_index94451 ------------------------------------------
chrom="chr6"
start=42022288
end=42022789
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(hg38,chrom,start,end,subtype="C24.21",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr2:46564850-46565200

In [ ]:
# orth_index94450 ------------------------------------------
chrom="chr6"
start=42025414
end=42025915
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(hg38,chrom,start,end,subtype="C24.21",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr2:46564850-46565200

In [ ]:
# orth_index65090 ------------------------------------------
chrom="chr16"
start=28930678
end=28931179
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(hg38,chrom,start,end,subtype="C24.21",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr2:46564850-46565200

#tree shrew sequence

In [ ]:
outdir="~/chrombpnet/Hs_TS_immune_chrombpnet/B_cell_Ery_ortho_predict/TS_B_pre_TS_B"

In [ ]:
TS_genome_fa = os.path.join("~/software/ref_tree_shrew/TreeShrew_scATAC_ref/fasta/genome.fa")

# output paths
#figout = os.path.join("~/chrombpnet/chromnpnet_cardiomyocytes/peak_from_ArchR_1000bp/chrombpnet_model_predicting/variant")

In [ ]:
TS_genome = Fasta(TS_genome_fa)

In [ ]:
# orth_index94458 ------------
chrom="chr3"
start=117496821
end=117497322
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(TS_genome,chrom,start,end,subtype="C24.21",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr17:2,239,095-2,241,207

In [ ]:
# orth_index94453 ------------------------------------------
chrom="chr3"
start=117454868
end=117455369
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(TS_genome,chrom,start,end,subtype="C24.21",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr2:46564850-46565200

In [ ]:
# orth_index94451 ------------------------------------------
chrom="chr3"
start=117441918
end=117442419
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(TS_genome,chrom,start,end,subtype="C24.21",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr2:46564850-46565200

In [ ]:
# orth_index94450 ------------------------------------------
chrom="chr3"
start=117438736
end=117439237
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(TS_genome,chrom,start,end,subtype="C24.21",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr2:46564850-46565200

In [ ]:
# orth_index65090 ------------------------------------------
chrom="chr2"
start=59849315
end=59849816
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(TS_genome,chrom,start,end,subtype="C24.21",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr2:46564850-46565200

红细胞

In [ ]:
proj_in="~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C24white_blood_cells/output/models"

In [ ]:
#load model
models = [
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C24.8/models/chrombpnet_nobias.h5")))
]

models_counts = [
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C24.8/models/chrombpnet_nobias.h5")))
]

human sequences

In [ ]:
outdir="~/chrombpnet/Hs_TS_immune_chrombpnet/B_cell_Ery_ortho_predict/TS_Ery_pre_Hs_Ery"

In [ ]:
# orth_index94462 ------------
chrom="chr6"
start=41957118
end=41957619
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(hg38,chrom,start,end,subtype="C24.8",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr17:2,239,095-2,241,207

In [ ]:
# orth_index94452 ------------
chrom="chr6"
start=42016971
end=42017472
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(hg38,chrom,start,end,subtype="C24.8",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr17:2,239,095-2,241,207

In [ ]:
# orth_index65100 ------------
chrom="chr16"
start=29009192
end=29009693
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(hg38,chrom,start,end,subtype="C24.8",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr17:2,239,095-2,241,207

### Tree shrew sequences


In [ ]:
outdir="~/chrombpnet/Hs_TS_immune_chrombpnet/B_cell_Ery_ortho_predict/TS_Ery_pre_TS_Ery"

In [ ]:
# orth_index94462 ------------
chrom="chr3"
start=117448428
end=117448929
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(TS_genome,chrom,start,end,subtype="C24.8",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr17:2,239,095-2,241,207

In [ ]:
# orth_index94452 ------------
chrom="chr3"
start=117518919
end=117519420
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(TS_genome,chrom,start,end,subtype="C24.8",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr17:2,239,095-2,241,207

In [ ]:
# orth_index65100  ------------
chrom="chr2"
start=59909670
end=59910171
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=False)
predict_sequence(TS_genome,chrom,start,end,subtype="C24.8",models=models,models_counts=models_counts,min_pos=807,max_pos=1308,show_fig=False)#chr17:2,239,095-2,241,207